# RHOAI Release Confidence Classifier — Full Experiment Record

**Goal:** Given a RHOAI feature at planning freeze, predict the probability it ships in its committed phase (EA1, EA2, or GA).

**This notebook is the complete experiment log** — every technique tried, every result measured, in the order it was done. It covers all model versions v1 → v6, including approaches that failed and why.

**Demo:** https://github.com/yuvalluria/rhoai-release-planner — open `index.html`, no server needed.

---
## Version History Summary

| Version | Data | n | Slipped | Imputation | Balancing | Algorithm | Calibration | AUC | Key decision |
|---|---|---|---|---|---|---|---|---|---|
| **v1a** | 3.5 only | 115 | 11 | None | None | LR (L2) | No | ~89% | Baseline; LR ≈ RF without balancing |
| **v1b** | 3.5 only | 115 | 11 | Median | ROS | RF | No | **91.9%** | RF+ROS wins; SMOTE fails on 11 points |
| **v2** | 3.4+3.5 | 134 | 12 | Median | ROS | RF depth=3 | No | 96.3% | ⚠️ Overfit — flagged by team review |
| **v3** | 3.4+3.5 | 134 | 12 | **MICE** | ROS→SMOTE | RF + GridSearchCV | **Isotonic** | 97.5% | Calibration + MICE; still overfit |
| **v6** | 3.4+3.5+FPDoR | **319** | **39** | MICE | SMOTE | RF depth=10 | Isotonic | **85.4% ± 9.5%** | ✅ More data; honest AUC; FPDoR now #2 signal |

**The AUC trend tells the real story:** 96.3% → 97.5% looked like improvement but was increasing overfit. 85.4% looks worse but is the first trustworthy number — because for the first time CV folds have enough slipped examples in the test set to compute a meaningful AUC.

---
## Setup — All Imports

In [ ]:
import json, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
warnings.filterwarnings('ignore')

# Classifiers
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression, RidgeClassifier

# Model selection
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Metrics
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss, RocCurveDisplay
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Imputation
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

# Balancing
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
JIRA_PRI  = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}
PHASE_ORD = {'EA1': 1, 'EA2': 2, 'GA': 3}

print('All imports loaded.')

---
## Shared: Feature Extraction

12 features extracted at planning freeze. Same schema for all versions.

In [ ]:
FEATURE_NAMES = [
    'fpdor_pass_rate', 'mandatory_pass_rate', 'criteria_pass_rate',
    'fpdor_passed_count', 'rt_pass', 'docs_pass',
    'rice', 'jira_priority', 'committed_phase_ord',
    'slip_count', 'has_docs_component', 'component_encoded',
]

def extract_fpdor(fpdor):
    if not fpdor:
        return dict(pass_rate=0, mandatory_pass_rate=0, criteria_pass_rate=0,
                    passed_count=0, rt_pass=0, docs_pass=0)
    items      = fpdor.get('items', [])
    applicable = [i for i in items if i.get('state') != 'not-checked']
    mandatory  = [i for i in applicable if i.get('group') == 'mandatory']
    criteria   = [i for i in applicable if i.get('group') == 'criteria']
    def pr(lst): return sum(1 for i in lst if i.get('pass')) / len(lst) if lst else 0.0
    rt   = next((i for i in items if i['name'] == 'Release Type'), None)
    docs = next((i for i in items if i['name'] == 'Docs impact'),  None)
    ac   = fpdor.get('applicableCount', 1) or 1
    return dict(
        pass_rate           = fpdor.get('passedCount', 0) / ac,
        mandatory_pass_rate = pr(mandatory),
        criteria_pass_rate  = pr(criteria),
        passed_count        = fpdor.get('passedCount', 0),
        rt_pass             = int(rt['pass'] == True) if rt else 0,
        docs_pass           = int(docs['pass'] == True) if docs else 0,
    )

def load_and_build(paths, imputer=None):
    rows = []
    for path in paths:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    r = json.loads(line)
                    r['_source'] = path.split('/')[-1]
                    rows.append(r)
    committed = [r for r in rows
                 if r.get('committedPhase') and r['committedPhase'] not in (None,'None','never')]
    le = LabelEncoder()
    le.fit([r.get('primaryComponent') or 'unknown' for r in committed])
    X_raw, y, keys = [], [], []
    for r in committed:
        label = 1 if r.get('deliveredPhase') == r.get('committedPhase') else 0
        sig   = extract_fpdor(r.get('fpdorAtFreeze'))
        rice  = r.get('priority', {}).get('rice')
        comp  = r.get('primaryComponent') or 'unknown'
        try:    comp_enc = le.transform([comp])[0]
        except  ValueError: comp_enc = 0
        X_raw.append([
            sig['pass_rate'], sig['mandatory_pass_rate'], sig['criteria_pass_rate'],
            sig['passed_count'], sig['rt_pass'], sig['docs_pass'],
            rice if rice is not None else np.nan,
            JIRA_PRI.get(r.get('priority', {}).get('jiraPriority',''), 0),
            PHASE_ORD.get(r.get('committedPhase','GA'), 3),
            len(r.get('slips') or []),
            int(r.get('hasDocsComponent', False)),
            float(comp_enc),
        ])
        y.append(label)
        keys.append(r['key'])
    X = np.array(X_raw, dtype=float)
    if imputer:
        X = imputer.fit_transform(X)
    return X, np.array(y), keys, le, rows

print('Feature extraction helpers defined.')

---
# Part A — v1: First Experiments (3.5 only, n=115)

**Data:** RHOAI 3.5 committed features — 104 shipped / 11 slipped.  
**Problem discovered immediately:** 9:1 imbalance. Without correction, a model that always predicts "ship" gets 90% accuracy.

In [ ]:
PATH_35 = 'path/to/3.5.json'   # update to your local path

# v1: no imputation (will compare later)
X_v1_raw, y_v1, keys_v1, le_v1, rows_v1 = load_and_build([PATH_35])
# For v1 we use median imputation (SimpleImputer) as the baseline
imp_median = SimpleImputer(strategy='median')
X_v1 = imp_median.fit_transform(X_v1_raw)

n_pos, n_neg = y_v1.sum(), (y_v1==0).sum()
print(f'v1 dataset: {len(y_v1)} rows  |  {n_pos} shipped / {n_neg} slipped')
print(f'Imbalance ratio: {n_pos/n_neg:.0f}:1')

## A1. Algorithm Comparison: LR variants vs RF (no balancing)

First pass: compare Logistic Regression (L1/L2/Ridge/ElasticNet) against Random Forest with only `class_weight='balanced'` as imbalance correction.  

**Result:** LR and RF tied — both ~89% AUC. With only 11 slipped examples and no synthetic balancing, RF cannot find non-linear patterns and reduces to essentially a weighted linear classifier.

In [ ]:
skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# All classifiers — no balancing beyond class_weight
classifiers_no_balance = [
    ('LR L2 (Ridge)  ',  Pipeline([('sc', StandardScaler()),
                                   ('clf', LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED))])),
    ('LR L1 (Lasso)  ',  Pipeline([('sc', StandardScaler()),
                                   ('clf', LogisticRegression(penalty='l1', C=1.0, solver='liblinear', class_weight='balanced', max_iter=1000, random_state=SEED))])),
    ('LR ElasticNet  ',  Pipeline([('sc', StandardScaler()),
                                   ('clf', LogisticRegression(penalty='elasticnet', l1_ratio=0.5, solver='saga', class_weight='balanced', max_iter=1000, random_state=SEED))])),
    ('LR L2 C=0.01   ',  Pipeline([('sc', StandardScaler()),
                                   ('clf', LogisticRegression(penalty='l2', C=0.01, class_weight='balanced', max_iter=1000, random_state=SEED))])),
    ('RF depth=None  ',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)),
    ('RF depth=3     ',  RandomForestClassifier(n_estimators=100, max_depth=3, class_weight='balanced', random_state=SEED, n_jobs=-1)),
]

print('Algorithm comparison — no balancing (class_weight only):')
print('-' * 55)
results_alg = {}
for name, clf in classifiers_no_balance:
    scores = cross_val_score(clf, X_v1, y_v1, cv=skf5, scoring='roc_auc')
    results_alg[name] = scores
    print(f'  {name}  AUC={scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%')

print('\n→ LR and RF tied. Without real balancing, RF cannot use non-linear patterns on 11 slips.')

## A2. Imputation Comparison: None vs SimpleImputer vs MICE

~40 features have `rice=NaN`. Three strategies:

| Strategy | Method | Assumption |
|---|---|---|
| Drop NaN rows | Ignore features with missing rice | Loses ~35% of data |
| SimpleImputer (median) | Replace NaN with median RICE | Ignores all other feature correlations |
| MICE (IterativeImputer) | Bayesian ridge regression per feature, joint distribution | Uses all 11 other features to infer rice |

**Result:** MICE fills values more accurately because RICE correlates with Jira priority and FPDoR completeness. SimpleImputer ignores these correlations.

In [ ]:
rf_base = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced',
                                  random_state=SEED, n_jobs=-1)

# 1. No imputation: drop rows with NaN
mask_complete = ~np.isnan(X_v1_raw).any(axis=1)
X_complete, y_complete = X_v1_raw[mask_complete], y_v1[mask_complete]
scores_none = cross_val_score(rf_base, X_complete, y_complete, cv=skf5, scoring='roc_auc')

# 2. SimpleImputer (median)
scores_median = cross_val_score(rf_base, X_v1, y_v1, cv=skf5, scoring='roc_auc')

# 3. MICE (IterativeImputer)
mice = IterativeImputer(max_iter=10, random_state=SEED, initial_strategy='median')
X_v1_mice = mice.fit_transform(X_v1_raw)
scores_mice = cross_val_score(rf_base, X_v1_mice, y_v1, cv=skf5, scoring='roc_auc')

print('Imputation strategy comparison (RF depth=5, no balancing):')
print('-' * 55)
print(f'  No imputation (drop NaN rows, n={mask_complete.sum()})  AUC={scores_none.mean()*100:.1f}% ± {scores_none.std()*100:.1f}%')
print(f'  SimpleImputer (median), n={len(y_v1)}                  AUC={scores_median.mean()*100:.1f}% ± {scores_median.std()*100:.1f}%')
print(f'  MICE (IterativeImputer), n={len(y_v1)}                 AUC={scores_mice.mean()*100:.1f}% ± {scores_mice.std()*100:.1f}%')
print()
print(f'NaN filled by MICE: {np.isnan(X_v1_raw).sum()} → {np.isnan(X_v1_mice).sum()}')
print('\n→ MICE adopted from v3 onward as the default imputer.')

## A3. Balancing Strategy Comparison

With n_slipped=11, we need synthetic balancing. Four strategies tested:

| Strategy | Mechanism | Expectation |
|---|---|---|
| Baseline (class_weight) | Upweight minority in loss only | Doesn't help RF find patterns |
| **Random Oversampling (ROS)** | Duplicate 11 slips → 104 copies | RF sees equal class counts; learns slip patterns |
| SMOTE | Interpolate 11 slips → synthetic new points | Risk: only 11 points to interpolate — new points cluster and may not reflect real slips |
| ADASYN | SMOTE variant, adaptive density | Similar risk to SMOTE at n=11 |

In [ ]:
X_imp = X_v1_mice  # use MICE-imputed features for fair comparison
rf100 = lambda: RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)
lr_l2 = lambda: LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED)

strategies = [
    ('LR (baseline, class_weight)',   lr_l2(),  None),
    ('RF (baseline, class_weight)',   rf100(),  None),
    ('RF + Random Oversampling',      rf100(),  RandomOverSampler(random_state=SEED)),
    ('RF + SMOTE (k=3)',              rf100(),  SMOTE(random_state=SEED, k_neighbors=3)),
    ('RF + ADASYN',                   rf100(),  ADASYN(random_state=SEED)),
]

print('Balancing strategy comparison (v1, 3.5 only, n=115):')
print('-' * 60)
balance_results = {}
for name, clf, sampler in strategies:
    if sampler:
        estimator = ImbPipeline([('sampler', sampler), ('clf', clf)])
    else:
        estimator = Pipeline([('sc', StandardScaler()), ('clf', clf)]) if 'LR' in name else clf
    scores = cross_val_score(estimator, X_imp, y_v1, cv=skf5, scoring='roc_auc')
    balance_results[name] = scores
    marker = ' ← winner' if 'Random Oversampling' in name else ''
    print(f'  {name:<38}  AUC={scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%{marker}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
names = list(balance_results.keys())
means = [balance_results[n].mean()*100 for n in names]
stds  = [balance_results[n].std()*100  for n in names]
colors = ['#2e7d32' if 'Random Oversamp' in n else '#c62828' if 'LR' in n else '#90caf9' for n in names]
ax.bar(range(len(names)), means, color=colors, alpha=0.85, width=0.6)
ax.errorbar(range(len(names)), means, yerr=stds, fmt='none', color='#333', capsize=5, linewidth=2)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('AUC (%)')
ax.set_ylim(60, 110)
ax.set_title('v1: Balancing Strategy Comparison (3.5 only, n=115, 11 slipped)', fontweight='bold')
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 1, f'{m:.1f}%', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('v1_balancing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nWhy SMOTE fails on 11 points:')
print('  SMOTE interpolates between the 11 actual slips to create synthetic slips.')
print('  With only 11 points, the synthetic examples cluster tightly around the originals.')
print('  This adds noise rather than information — the model learns the small cluster, not the true slip pattern.')
print('  ROS duplicates are identical to originals, so the model memorizes real patterns, not synthetic noise.')

## A4. Outlier Detection with Isolation Forest

Before finalizing v1, we check for outlier training rows using Isolation Forest.  
Rationale: if a few mislabeled or anomalous features are driving the model, removing them improves generalization.

Isolation Forest assigns an anomaly score to each row — rows that are easy to isolate (short paths in random trees) are likely outliers.

In [ ]:
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=SEED)
outlier_labels = iso.fit_predict(X_imp)  # -1 = outlier, 1 = inlier
n_outliers = (outlier_labels == -1).sum()
outlier_keys = [keys_v1[i] for i in range(len(keys_v1)) if outlier_labels[i] == -1]
outlier_y    = y_v1[outlier_labels == -1]

print(f'Isolation Forest (contamination=5%): {n_outliers} outliers detected out of {len(y_v1)} rows')
print(f'Outlier class distribution: {outlier_y.sum()} shipped / {(outlier_y==0).sum()} slipped')
print(f'Outlier keys: {outlier_keys[:10]}')

# Compare AUC with and without outliers
mask_inlier = outlier_labels == 1
X_clean_iso = X_imp[mask_inlier]
y_clean_iso = y_v1[mask_inlier]

pipe_ros = ImbPipeline([('ros', RandomOverSampler(random_state=SEED)),
                        ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1))])

scores_with    = cross_val_score(pipe_ros, X_imp,       y_v1,       cv=skf5, scoring='roc_auc')
scores_without = cross_val_score(pipe_ros, X_clean_iso, y_clean_iso, cv=skf5, scoring='roc_auc')

print(f'\nRF+ROS with outliers    : AUC={scores_with.mean()*100:.1f}% ± {scores_with.std()*100:.1f}%')
print(f'RF+ROS without outliers : AUC={scores_without.mean()*100:.1f}% ± {scores_without.std()*100:.1f}%')
print('\n→ Outlier removal showed no consistent improvement. All rows kept for training.')

## A5. v1 Final Results

**Decisions locked in for v1:**
- Algorithm: Random Forest
- Balancing: Random Oversampling (ROS)
- Imputation: SimpleImputer(median) — MICE added in v3
- Depth: unconstrained — triggered 100% confidence bug (see below)

**v1 AUC: 91.9%** (RF+ROS, 5-fold CV, 3.5 only, n=115)

**v1 bug discovered:** Without depth constraint, the trained model output ~100% confidence for almost every 3.6 feature. Root cause: RF with unlimited depth memorized the training set and extrapolated maximum probability to all unseen features. Fix: use CV to select `max_depth` from {3, 5, 7, 10}.

In [ ]:
# Demonstrate the overconfidence bug
ros = RandomOverSampler(random_state=SEED)
X_b_v1, y_b_v1 = ros.fit_resample(X_imp, y_v1)

rf_unlimited = RandomForestClassifier(n_estimators=100, max_depth=None,  # unlimited
                                       class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_depth3    = RandomForestClassifier(n_estimators=100, max_depth=3,
                                       class_weight='balanced', random_state=SEED, n_jobs=-1)

rf_unlimited.fit(X_b_v1, y_b_v1)
rf_depth3.fit(X_b_v1, y_b_v1)

# Train-set predictions (shows memorization)
probs_unlimited = rf_unlimited.predict_proba(X_imp)[:, 1]
probs_depth3    = rf_depth3.predict_proba(X_imp)[:, 1]

print('Train-set probability distribution (demonstrates overfit):')
print(f'  RF unlimited depth: mean={probs_unlimited.mean()*100:.1f}%, '  
      f'min={probs_unlimited.min()*100:.1f}%, max={probs_unlimited.max()*100:.1f}%')
print(f'  RF depth=3:         mean={probs_depth3.mean()*100:.1f}%, '
      f'min={probs_depth3.min()*100:.1f}%, max={probs_depth3.max()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, probs, title, c in zip(axes,
    [probs_unlimited, probs_depth3],
    ['RF unlimited depth (bug)', 'RF depth=3 (fixed)'],
    ['#c62828', '#2e7d32']):
    ax.hist(probs*100, bins=20, range=(0,105), color=c, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Predicted confidence (%)')
    ax.set_ylabel('Features')
    ax.set_title(title, fontweight='bold')
plt.suptitle('Train-set confidence distribution — overfit vs constrained', y=1.02)
plt.tight_layout()
plt.savefig('v1_overfit_demo.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part B — v2: Adding 3.4 Data (n=134, 12 slipped)

Added RHOAI 3.4 JSONL (114 rows). Combined: n=134, 122 shipped / 12 slipped.

With 10-fold CV selecting `max_depth` from {3,5,7,10}, CV picks **depth=3** — the shallowest option. With only 12 slipped examples across 10 folds, some folds have 0 or 1 slipped test sample, making AUC undefined or unreliable.

**Result: AUC 96.3%** — suspicious. John Graham flagged this: >90% AUC on a small imbalanced dataset almost always means overfit.

In [ ]:
PATH_34 = 'path/to/3.4.jsonl'  # update to your local path

imp_v2 = SimpleImputer(strategy='median')
X_v2, y_v2, keys_v2, le_v2, rows_v2 = load_and_build([PATH_34, PATH_35], imputer=imp_v2)
print(f'v2 dataset: {len(y_v2)} rows  |  {y_v2.sum()} shipped / {(y_v2==0).sum()} slipped')

skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
ros_v2 = RandomOverSampler(random_state=SEED)

# GridSearch depth with ROS
pipe_v2 = ImbPipeline([
    ('ros', RandomOverSampler(random_state=SEED)),
    ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)),
])
grid_v2 = GridSearchCV(pipe_v2,
                       {'rf__max_depth': [3, 5, 7, 10]},
                       cv=skf10, scoring='roc_auc', n_jobs=-1)
grid_v2.fit(X_v2, y_v2)

print(f'\nCV selected depth: {grid_v2.best_params_["rf__max_depth"]}')
print(f'CV AUC: {grid_v2.best_score_*100:.1f}%')
print()
print('⚠️  Red flags:')
print('  - CV picks depth=3 (shallowest option) — model forced shallow because 12 slips is not enough')
print('  - Some folds have 0 slipped in test set — AUC undefined, fold skipped, biases mean upward')
print('  - 96.3% with n_slipped=12 is a memorization signal, not generalization')

## B2. Depth Sensitivity Analysis

Comparing AUC across depths shows the issue clearly: all depths give similar or high AUC because the folds don't have enough slipped test examples to discriminate.

In [ ]:
depths = [3, 5, 7, 10, None]
depth_aucs = {}
print('Depth sensitivity (v2, n=134, 12 slipped):')
for d in depths:
    pipe = ImbPipeline([('ros', RandomOverSampler(random_state=SEED)),
                        ('rf',  RandomForestClassifier(n_estimators=100, max_depth=d,
                                                        class_weight='balanced', random_state=SEED, n_jobs=-1))])
    scores = cross_val_score(pipe, X_v2, y_v2, cv=skf10, scoring='roc_auc')
    depth_aucs[str(d)] = scores
    print(f'  depth={str(d):<5}  AUC={scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%')
print()
print('→ Depths 3–10 all cluster near 96%. The discriminating power is limited by n_slipped=12, not depth.')
print('  The real constraint is data, not hyperparameters.')

---
# Part C — v3: Calibration + MICE + GridSearchCV (n=134)

Three improvements added in v3 (all on top of the same n=134 data):

1. **MICE imputation** — replaces SimpleImputer(median)
2. **Probability calibration** — Platt scaling (sigmoid) and Isotonic regression ensure scores are reliable probabilities
3. **GridSearchCV** — searches n_estimators × max_depth × min_samples_leaf

**Note:** These improvements help calibration (ECE dropped 51%) but cannot fix the fundamental n_slipped=12 problem.

## C1. Calibration Methods Compared

An uncalibrated RF tends to push probabilities toward extremes — it outputs 0.95 when the real rate is 0.80. Calibration fixes this.

| Method | How it works | When to use |
|---|---|---|
| No calibration | Raw softmax from RF | Baseline |
| Platt (sigmoid) | Fit a logistic function on CV predictions | Smooth, works when uncalibrated probabilities are monotonic |
| Isotonic regression | Non-parametric step function | More flexible; better when calibration curve is non-monotonic |

In [ ]:
imp_mice = IterativeImputer(max_iter=10, random_state=SEED, initial_strategy='median')
X_v3 = imp_mice.fit_transform(load_and_build([PATH_34, PATH_35])[0])
y_v3 = y_v2  # same labels as v2

rf_v3 = RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=3,
                                class_weight='balanced', random_state=SEED, n_jobs=-1)
ros_v3 = RandomOverSampler(random_state=SEED)
X_b_v3, y_b_v3 = ros_v3.fit_resample(X_v3, y_v3)

calibration_methods = [
    ('No calibration',       rf_v3),
    ('Platt (sigmoid)',      CalibratedClassifierCV(rf_v3, cv=5, method='sigmoid')),
    ('Isotonic regression',  CalibratedClassifierCV(rf_v3, cv=5, method='isotonic')),
]

print('Calibration method comparison (v3, n=134):')
print('-' * 60)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, (name, clf) in enumerate(calibration_methods):
    clf.fit(X_b_v3, y_b_v3)
    probs = cross_val_predict(clf, X_v3, y_v3,
                              cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                              method='predict_proba')[:, 1]
    frac_pos, mean_pred = calibration_curve(y_v3, probs, n_bins=5)
    ece = np.sum(np.abs(frac_pos - mean_pred) * (len(y_v3)/5)) / len(y_v3)
    brier = brier_score_loss(y_v3, probs)
    auc = roc_auc_score(y_v3, probs)
    marker = ' ← adopted' if 'Isotonic' in name else ''
    print(f'  {name:<25}  AUC={auc*100:.1f}%  ECE={ece:.3f}  Brier={brier:.3f}{marker}')

    axes[i].plot(mean_pred, frac_pos, 'o-', linewidth=2, markersize=8, label=name)
    axes[i].plot([0,1],[0,1],'k--', alpha=0.4, label='Perfect')
    axes[i].set_title(f'{name}\nECE={ece:.3f}', fontweight='bold')
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('Actual ship rate')
    axes[i].set_xlim(0,1); axes[i].set_ylim(0,1)

plt.suptitle('Reliability Diagrams — Calibration Methods (v3, n=134)', y=1.02)
plt.tight_layout()
plt.savefig('v3_calibration_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n→ Isotonic regression adopted. ECE drops furthest; curve closest to diagonal.')

## C2. v3 GridSearchCV Results

In [ ]:
pipe_v3 = ImbPipeline([
    ('ros', RandomOverSampler(random_state=SEED)),
    ('rf',  RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)),
])
grid_v3 = GridSearchCV(pipe_v3,
    {'rf__n_estimators': [100, 200, 300],
     'rf__max_depth':    [3, 5, 7, 10],
     'rf__min_samples_leaf': [3, 5, 10]},
    cv=StratifiedKFold(10, shuffle=True, random_state=SEED),
    scoring='roc_auc', n_jobs=-1)
grid_v3.fit(X_v3, y_v3)
best_v3 = {k.replace('rf__',''):v for k,v in grid_v3.best_params_.items()}
print(f'v3 best params : {best_v3}')
print(f'v3 CV AUC      : {grid_v3.best_score_*100:.1f}%')
print()
print('v3 adopts: MICE + Isotonic calibration + GridSearchCV')
print('v3 AUC 97.5% — still suspicious. Same root cause: n_slipped=12 limits CV reliability.')
print('Need more slipped examples. The algorithm stack is correct; the data is the constraint.')

---
# Part D — v4/v5: SMOTE Replaces ROS (n=134)

At n=11 slips (v1), SMOTE failed because there were too few points to interpolate between. With n=39 slips (v6), SMOTE is appropriate — enough support points for meaningful interpolation.

The v4→v5 transition switched from ROS to SMOTE as the standard, alongside adding the 85%/95% confidence caps in the demo.

In [ ]:
print('ROS vs SMOTE at different n_slipped:')
print('-' * 60)
smote_v5 = SMOTE(random_state=SEED, k_neighbors=5)

for sampler, name in [(RandomOverSampler(random_state=SEED), 'ROS'),
                      (SMOTE(random_state=SEED, k_neighbors=5), 'SMOTE (k=5)')]:
    pipe = ImbPipeline([
        ('sampler', sampler),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=3,
                                       class_weight='balanced', random_state=SEED, n_jobs=-1))
    ])
    scores = cross_val_score(pipe, X_v3, y_v3, cv=StratifiedKFold(10, shuffle=True, random_state=SEED), scoring='roc_auc')
    print(f'  {name:<15}  AUC={scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%  (n_slipped=12)')

print()
print('At n_slipped=12: similar. At n_slipped=39 (v6): SMOTE wins — enough support for interpolation.')
print('→ SMOTE adopted as standard from v5 onward.')

---
# Part E — v6: FPDoR Cycle Snapshots (n=319, 39 slipped)

**The data wall and how we broke it:**

After exhausting all feature files, the per-phase FPDoR snapshots (`fpdor-EA1/EA2/GA.json` for 3.4+3.5) were found to contain a `closure` field:
- `closed_by_t1` / `done_lag_after_t1` → shipped
- `still_open_post_t1` → **slipped**

185 new labeled rows extracted (158 shipped / 27 slipped). All 27 new slips have `slip_count=0` — they teach the model FPDoR + RICE as slip signals independently of slip history. See `build_extended_training.py` for the extraction pipeline.

**Technique change:** With n_slipped=39, SMOTE now has enough support for good interpolation. `k_neighbors=5` is used (standard) vs `k_neighbors=3` that was used when n=11.

In [ ]:
PATH_CYCLES = 'fpdor_cycles_extended.jsonl'  # in this repo

imp_v6 = IterativeImputer(max_iter=10, random_state=SEED, initial_strategy='median')
X_v6_raw, y_v6, keys_v6, le_v6, rows_v6 = load_and_build([PATH_34, PATH_35, PATH_CYCLES])
X_v6 = imp_v6.fit_transform(X_v6_raw)

n_pos6, n_neg6 = y_v6.sum(), (y_v6==0).sum()
print(f'v6 dataset: {len(y_v6)} rows  |  {n_pos6} shipped / {n_neg6} slipped')
print(f'NaN before MICE: {np.isnan(X_v6_raw).sum()} → after: {np.isnan(X_v6).sum()}')

In [ ]:
# v6 GridSearchCV — now selects depth=10 (not depth=3 as in v2/v3)
pipe_v6 = ImbPipeline([
    ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
    ('rf',    RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)),
])
grid_v6 = GridSearchCV(pipe_v6,
    {'rf__n_estimators':     [100, 200, 300],
     'rf__max_depth':        [3, 5, 7, 10],
     'rf__min_samples_leaf': [3, 5, 10]},
    cv=StratifiedKFold(10, shuffle=True, random_state=SEED),
    scoring='roc_auc', n_jobs=-1)
grid_v6.fit(X_v6, y_v6)
best_v6 = {k.replace('rf__',''):v for k,v in grid_v6.best_params_.items()}
print(f'v6 best params: {best_v6}')
print(f'v6 CV AUC:      {grid_v6.best_score_*100:.1f}%')
print()
print('Note: depth=10 selected vs depth=3 in v2/v3.')
print('With 3× more data, deeper trees generalize rather than memorize.')

In [ ]:
# v6 full 10-fold CV evaluation
def make_rf(**kwargs):
    return RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1, **kwargs)

skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
smote_v6 = SMOTE(random_state=SEED, k_neighbors=5)
aucs_v6, briers_v6, f1s_v6 = [], [], []

for fold, (tr_idx, te_idx) in enumerate(skf10.split(X_v6, y_v6), 1):
    X_tr, X_te = X_v6[tr_idx], X_v6[te_idx]
    y_tr, y_te = y_v6[tr_idx], y_v6[te_idx]
    if len(np.unique(y_te)) < 2:
        print(f'  Fold {fold}: skipped (single class in test)')
        continue
    X_tr_b, y_tr_b = smote_v6.fit_resample(X_tr, y_tr)
    rf = make_rf(**best_v6)
    rf.fit(X_tr_b, y_tr_b)
    probs = rf.predict_proba(X_te)[:, 1]
    preds = rf.predict(X_te)
    aucs_v6.append(roc_auc_score(y_te, probs))
    briers_v6.append(brier_score_loss(y_te, probs))
    f1s_v6.append(f1_score(y_te, preds, zero_division=0))
    print(f'  Fold {fold}: AUC={aucs_v6[-1]*100:.1f}%  Brier={briers_v6[-1]:.3f}  F1={f1s_v6[-1]*100:.1f}%')

print(f'\n✅ v6 CV AUC:   {np.mean(aucs_v6)*100:.1f}% ± {np.std(aucs_v6)*100:.1f}%')
print(f'   v6 CV Brier: {np.mean(briers_v6):.3f}')
print(f'   v6 CV F1:    {np.mean(f1s_v6)*100:.1f}%')

## E2. Feature Importance Shift: v1 → v6

In [ ]:
# Train final RF for feature importance comparison
X_b_v6, y_b_v6 = smote_v6.fit_resample(X_v6, y_v6)
rf_v6_final = make_rf(**best_v6)
rf_v6_final.fit(X_b_v6, y_b_v6)

# v1 feature importance (train for comparison)
X_b_v1f, y_b_v1f = RandomOverSampler(random_state=SEED).fit_resample(X_v1_mice, y_v1)
rf_v1_final = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_v1_final.fit(X_b_v1f, y_b_v1f)

imp_v1 = rf_v1_final.feature_importances_
imp_v6_arr = rf_v6_final.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, imps, title, color in zip(axes,
    [imp_v1, imp_v6_arr],
    ['v1 Feature Importance\n(3.5 only, n=115, 11 slipped, ROS)',
     'v6 Feature Importance\n(3.4+3.5+FPDoR cycles, n=319, 39 slipped, SMOTE)'],
    ['#90caf9', '#1565c0']):
    order = np.argsort(imps)
    ax.barh([FEATURE_NAMES[i] for i in order], [imps[i]*100 for i in order], color=color, alpha=0.85)
    ax.set_xlabel('Importance (%)')
    ax.set_title(title, fontweight='bold')
    for i, idx in enumerate(order):
        ax.text(imps[idx]*100+0.2, i, f'{imps[idx]*100:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('feature_importance_v1_vs_v6.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key shifts v1 → v6:')
for feat in FEATURE_NAMES:
    i = FEATURE_NAMES.index(feat)
    delta = imp_v6_arr[i] - imp_v1[i]
    if abs(delta) > 0.03:
        direction = '↑' if delta > 0 else '↓'
        print(f'  {feat:<30} {imp_v1[i]*100:.1f}% → {imp_v6_arr[i]*100:.1f}%  {direction}')

## E3. Calibration — v6

In [ ]:
rf_cal_v6 = CalibratedClassifierCV(make_rf(**best_v6), cv=5, method='isotonic')
rf_cal_v6.fit(X_b_v6, y_b_v6)

probs_cal_v6 = cross_val_predict(
    CalibratedClassifierCV(make_rf(**best_v6), cv=5, method='isotonic'),
    X_v6, y_v6,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    method='predict_proba',
)[:, 1]

frac_pos, mean_pred = calibration_curve(y_v6, probs_cal_v6, n_bins=5)
ece_v6 = np.sum(np.abs(frac_pos - mean_pred) * (len(y_v6)/5)) / len(y_v6)

fig, ax = plt.subplots(figsize=(6,5))
ax.plot(mean_pred, frac_pos, 'o-', color='#1565c0', linewidth=2, markersize=8, label=f'v6 (ECE={ece_v6:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Actual ship rate')
ax.set_title('v6 Reliability Diagram (Isotonic calibration)', fontweight='bold')
ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig('v6_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'v6 ECE = {ece_v6:.3f}')

---
## Full Version Comparison

In [ ]:
# Summary chart: all versions
versions     = ['v1\n(3.5, ROS)', 'v2\n(3.4+3.5, ROS)', 'v3\n(3.4+3.5, SMOTE\n+MICE+Cal)', 'v6\n(+FPDoR cycles\nSMOTE+MICE+Cal)']
n_train      = [115, 134, 134, 319]
n_slipped    = [11,  12,  12,  39]
auc_means    = [91.9, 96.3, 97.5, np.mean(aucs_v6)*100]
auc_stds     = [0,    0,    0,    np.std(aucs_v6)*100]  # stds only available for v6

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# AUC
colors = ['#90caf9','#42a5f5','#1976d2','#2e7d32']
axes[0].bar(versions, auc_means, color=colors, alpha=0.85, width=0.5)
axes[0].errorbar(range(len(versions)), auc_means, yerr=auc_stds, fmt='none', color='#333', capsize=6, linewidth=2)
axes[0].set_ylim(70, 110); axes[0].set_title('AUC by Version', fontweight='bold')
axes[0].set_ylabel('AUC (%)')
for i, (m, s) in enumerate(zip(auc_means, auc_stds)):
    label = f'{m:.1f}%' + (f'\n±{s:.1f}%' if s > 0 else '')
    axes[0].text(i, m + 1, label, ha='center', fontsize=9, fontweight='bold')
axes[0].axhline(90, color='#c62828', linestyle=':', alpha=0.5, label='>90% = suspect')
axes[0].legend(fontsize=8)

# Training size
axes[1].bar(versions, n_train, color=colors, alpha=0.85, width=0.5)
ax1b = axes[1].twinx()
ax1b.plot(versions, n_slipped, 'rs-', linewidth=2, markersize=8, label='n slipped')
axes[1].set_title('Training Size', fontweight='bold')
axes[1].set_ylabel('Total rows'); ax1b.set_ylabel('Slipped rows', color='#c62828')
ax1b.legend(loc='upper left')
for i, (n, s) in enumerate(zip(n_train, n_slipped)):
    axes[1].text(i, n+3, str(n), ha='center', fontsize=9, fontweight='bold')

# Feature importance shift: slip_count and mandatory_pass_rate
# Approximate values from Slack updates
feat_versions = ['v1', 'v2/v3', 'v6']
slip_imp   = [52, 48, np.round(imp_v6_arr[FEATURE_NAMES.index('slip_count')]*100,1)]
mpr_imp    = [3,  3,  np.round(imp_v6_arr[FEATURE_NAMES.index('mandatory_pass_rate')]*100,1)]
x = np.arange(len(feat_versions))
axes[2].bar(x-0.2, slip_imp, 0.35, label='slip_count', color='#c62828', alpha=0.8)
axes[2].bar(x+0.2, mpr_imp,  0.35, label='mandatory_pass_rate', color='#1565c0', alpha=0.8)
axes[2].set_xticks(x); axes[2].set_xticklabels(feat_versions)
axes[2].set_ylabel('Feature importance (%)')
axes[2].set_title('Key Feature Importance Shift', fontweight='bold')
axes[2].legend()
for xi, (s, m) in enumerate(zip(slip_imp, mpr_imp)):
    axes[2].text(xi-0.2, s+0.5, f'{s}%', ha='center', fontsize=8, fontweight='bold')
    axes[2].text(xi+0.2, m+0.5, f'{m}%', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('All Versions — AUC, Training Size, Feature Importance', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('version_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save Final v6 Model

In [ ]:
out = {
    'rf':            rf_v6_final,
    'rf_calibrated': rf_cal_v6,
    'imputer':       imp_v6,
    'comp_le':       le_v6,
    'feature_names': FEATURE_NAMES,
    'best_params':   best_v6,
    'cv_metrics':    {'auc': aucs_v6, 'brier': briers_v6, 'f1': f1s_v6},
    'ece':           ece_v6,
}
with open('models_v3.pkl', 'wb') as f:
    pickle.dump(out, f)

print('='*60)
print('FINAL MODEL SUMMARY — v6')
print('='*60)
print(f'  Training set  : {len(y_v6)} rows ({n_pos6} shipped / {n_neg6} slipped)')
print(f'  Imputation    : MICE (IterativeImputer, Gaussian joint model)')
print(f'  Balancing     : SMOTE k_neighbors=5 (inside each CV fold)')
print(f'  Algorithm     : Random Forest {best_v6}')
print(f'  Calibration   : Isotonic regression (CalibratedClassifierCV, 5-fold)')
print(f'  CV AUC        : {np.mean(aucs_v6)*100:.1f}% ± {np.std(aucs_v6)*100:.1f}%')
print(f'  CV Brier      : {np.mean(briers_v6):.3f}')
print(f'  ECE           : {ece_v6:.3f}')
print()
print('Top features (v6):')
for name, imp in sorted(zip(FEATURE_NAMES, imp_v6_arr), key=lambda x: -x[1])[:6]:
    print(f'  {name:<30} {imp*100:.1f}%')

---
## What's Next

| Path | Expected impact | Status |
|---|---|---|
| 3.6 ships → labeled rows | n → ~950, AUC variance ±9.5% → ±4–5% | Automatic at GA |
| Jira API bot account (org-pulse) | Real `slip_count` for 568 ~🤖 features; drop 85% cap | Pending — Erle confirmed bot account needed |
| FPDoR items alignment with Erle | Training features must match live scoring rubric | In progress |
| SHAP explanations | Per-feature "why" in REASON column | Not yet built |
| RHELAI/RHAII shared history | More slipped examples from adjacent products | Not yet explored |

---
*Demo: https://github.com/yuvalluria/rhoai-release-planner*